# Step 04.1 — Linear：从矩阵乘法到可学习层（整合版）

这一版整合了此前 4.1 的三个版本。目标不是只记住 `nn.Linear` 的 API，而是从你已经会的矩阵乘法出发，完整理解：

1. Linear 为什么能产生“新特征”；
2. 权重矩阵的 shape 为什么这样设计；
3. `(2,)`、`(1,2)`、`(2,1)` 为什么不同；
4. 多个样本为什么共享同一套权重；
5. `nn.Linear` / `layer` 到底封装了什么。

> 核心结论：`nn.Linear` 的数学核心仍然是矩阵乘法；神经网络训练只是让其中的权重不再由我们手写，而是从数据中逐步学出来。

## 1. 先手算 Linear：它为什么叫“特征混合”

先完全不用 `mlx.nn`。

假设一个输入只有两个特征：

```text
x = [2, -1]
```

我们希望把它变成三个输出特征。于是准备三组权重，每一组都读取输入的两个特征：

```text
W =
[[1, 0],
 [0, 1],
 [1, 1]]
```

`W.shape = (3,2)` 的意义不是背公式，而是：

```text
3 个输出特征
×
每个输出都需要 2 个输入权重
=
(3,2)
```

三个输出分别是：

```text
y₀ = 1*x₀ + 0*x₁
y₁ = 0*x₀ + 1*x₁
y₂ = 1*x₀ + 1*x₁
```

所以 Linear 的一个重要意义就是：**新的每个特征都可以是旧特征的加权组合。**

In [ ]:
import mlx.core as mx

x = mx.array([2.0, -1.0])

W = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

print("x      :", x, "shape =", x.shape)
print("W shape:", W.shape)
print("W.T    :", W.T.shape)

如果我们暂时按深度学习常用的 **features-last** 写法，把输入放在左边，那么计算写成：

```python
y = x @ W.T
```

手算：

- 第 1 个输出：`2×1 + (-1)×0 = 2`
- 第 2 个输出：`2×0 + (-1)×1 = -1`
- 第 3 个输出：`2×1 + (-1)×1 = 1`

所以：

```text
[2, -1] -> [2, -1, 1]
```

注意第三个输出 `[1,1]` 同时读取两个输入，所以它不是简单复制原特征，而是在做真正的**feature mixing（特征混合）**。

In [ ]:
y = x @ W.T
print("y      :", y)
print("y shape:", y.shape)

## 2. `(2,)`、`(1,2)`、`(2,1)`：一维向量不是行矩阵

这里必须补一个之前容易跳过的前置知识。

同样两个数字 `[2,-1]` 可以有三种不同 shape：

```text
(2,)   -> 1D vector
(1,2)  -> 1×2 row matrix
(2,1)  -> 2×1 column matrix
```

`(2,)` 只有一个 axis，它本身并没有“行”和“列”。

In [ ]:
x_row = x.reshape(1, 2)
x_col = x.reshape(2, 1)

print("x     :", x, "shape =", x.shape)
print("x_row :", x_row, "shape =", x_row.shape)
print("x_col :")
print(x_col)
print("shape =", x_col.shape)

### 两种常见写法其实是同一件事

在线性代数课里，常把输入写成列向量：

```text
W      @ x_col -> y_col
(3,2)   (2,1)    (3,1)
```

这就是经典的：

```text
y = W x
```

而深度学习框架通常把 feature 放在最后一个 axis，因此显式二维情况下更常写：

```text
x_row @ W.T -> y_row
(1,2)  (2,3)   (1,3)
```

两种写法得到的三个数完全相同，只是摆放方向不同。

In [ ]:
y_col = W @ x_col
y_row = x_row @ W.T

print("column form:")
print(y_col, "shape =", y_col.shape)

print("\nrow form:")
print(y_row, "shape =", y_row.shape)

### 那 `(2,) @ (2,3)` 为什么也合法？

现在回到：

```python
x @ W.T
```

此时：

```text
x.shape   = (2,)
W.T.shape = (2,3)
```

`@` 对 **1D 左操作数**有特殊规则。理解时可以把 `(2,)` 临时看成 `(1,2)`：

```text
(2,)  -> 临时按 (1,2) 参与计算
(1,2) @ (2,3) -> (1,3)
```

乘法完成后，刚才临时加入的 size-1 维度会被去掉，因此真正返回：

```text
(3,)
```

重要：**变量 `x` 本身从未真的变成 `(1,2)`。**

In [ ]:
y_1d = x @ W.T
y_explicit_row = x.reshape(1, 2) @ W.T

print("1D rule       :", y_1d, "shape =", y_1d.shape)
print("explicit row  :", y_explicit_row, "shape =", y_explicit_row.shape)
print("original x    :", x.shape)

## 3. 多个样本为什么共用同一套 `W`？

神经网络不会只处理一个向量。假设一次有两个样本：

```text
X =
[[ 2, -1],
 [ 3,  4]]

X.shape = (2,2)
```

这里第一维的 `2` 表示两个样本；每个样本最后一维仍有两个特征。

计算：

```text
(2,2) @ (2,3) -> (2,3)
```

意味着两行输入分别经过**同一套 `W`**。

这叫 parameter sharing：模型不是为每个样本重新造一套 Linear 权重，而是让同一个变换规则应用到所有样本。

In [ ]:
X = mx.array([
    [2.0, -1.0],
    [3.0,  4.0],
])

Y = X @ W.T

print("X shape  :", X.shape)
print("W.T shape:", W.T.shape)
print("Y shape  :", Y.shape)
print(Y)

以后 MiniGPT 的 hidden state 通常是：

```text
(B, T, C)
```

Linear 仍然主要作用在最后的 feature 维：

```text
(B, T, C_in) -> Linear -> (B, T, C_out)
```

Batch 和 Time 维被保留；每个位置都复用同一套 Linear 参数。

## 4. `nn.Linear`、`layer` 与 `mlx.nn` 到底是什么？

先导入：

```python
import mlx.nn as nn
```

`mlx.nn` 是 MLX 提供的 neural-network module library；我们把它简写成 `nn`。

然后：

```python
layer = nn.Linear(2, 3, bias=False)
```

右边不是在立即处理 `x`，而是在**创建一个 Linear 层对象**，再把这个对象保存到变量 `layer`。

可以暂时把它想成一台小机器：

```text
layer
├── weight        <- 当前保存的权重矩阵
├── bias          <- 可选，这里关闭
└── forward rule  <- 收到输入后如何计算输出
```

因为我们写的是：

```python
nn.Linear(2, 3, bias=False)
```

所以 MLX 会创建：

```text
layer.weight.shape = (3,2)
```

这些 weight 初始是随机值。当前具体数字还没有任务语义；后面训练的核心就是根据 loss 和 gradient 不断修改它们。

In [ ]:
import mlx.nn as nn

layer = nn.Linear(2, 3, bias=False)

print("object type :", type(layer))
print("weight shape:", layer.weight.shape)
print("weight      :")
print(layer.weight)

### 为什么 `layer(x)` 看起来像函数？

`layer` 是一个**对象（object）**，不是普通函数。

Python 允许对象定义 `__call__` 行为，因此某些对象可以像函数一样写：

```python
out = layer(x)
```

这里的意思是：**让这个 Linear 对象用它当前保存的参数处理输入 `x`。**

要严格区分：

```python
layer = nn.Linear(...)
```

= 创建层对象和它的参数；

而：

```python
layer(x)
```

= 调用这层的前向计算。

In [ ]:
out_1d = layer(x)
out_batch = layer(X)
mx.eval(out_1d, out_batch)

print("1D input :", x.shape, "->", out_1d.shape)
print("2D input :", X.shape, "->", out_batch.shape)

## 5. 最后拆掉黑盒：验证 `layer(X)` 就是矩阵乘法

因为这里设置：

```python
bias=False
```

所以对这个 Linear 层，概念上就是：

```python
layer(X) == X @ layer.weight.T
```

现在直接同时计算两边，看它们是否一致。

In [ ]:
from_layer = layer(X)
manual = X @ layer.weight.T

mx.eval(from_layer, manual)

print("layer(X):")
print(from_layer)

print("\nmanual X @ weight.T:")
print(manual)

print("\nsame:", bool(mx.allclose(from_layer, manual)))

如果最后输出：

```text
same: True
```

你就已经亲手证明了：至少在 `bias=False` 时，`nn.Linear` 并没有隐藏新的数学魔法。

### 4.1 最终结论

- Linear 的底层核心是矩阵乘法；
- `W.shape = (output_features, input_features)`；
- 一个输出特征是所有输入特征的一次加权组合，因此 Linear 可以做 feature mixing；
- `(2,)` 是 1D vector，不等于 `(1,2)` 或 `(2,1)`；
- `1D @ 2D` 有特殊的 matmul 语义，但不会永久修改原向量 shape；
- 多个样本共享同一个 `W`；
- `nn.Linear(...)` 创建的是一个保存参数和计算规则的对象；
- `layer(x)` 是调用这个对象的前向计算；
- `bias=False` 时，可以直接验证 `layer(X) == X @ layer.weight.T`；
- 后续训练要做的，就是把随机初始化的 `weight` 学成对任务有用的权重。

下一节 4.2 再正式学习：**Parameter、bias 和 `nn.Module`。**